<a href="https://colab.research.google.com/github/robertkigobe/llm_engineering/blob/main/cds_pipelines.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Hardware specifications

In [1]:
# Let's check the GPU - it should be a Tesla T4

gpu_info = !nvidia-smi
gpu_info = '\n'.join(gpu_info)
if gpu_info.find('failed') >= 0:
  print('Not connected to a GPU')
else:
  print(gpu_info)
  if gpu_info.find('Tesla T4') >= 0:
    print("Success - Connected to a T4")
  else:
    print("NOT CONNECTED TO A T4")

zsh:1: command not found: nvidia-smi
NOT CONNECTED TO A T4


# Setup & installs

In [2]:

!pip -q install -U \
  pandas \
  gradio \
  transformers \
  accelerate \
  sentencepiece


ERROR: Exception:
Traceback (most recent call last):
  File "/Applications/anaconda3/lib/python3.8/site-packages/pip/_internal/cli/base_command.py", line 106, in _run_wrapper
    status = _inner_run()
  File "/Applications/anaconda3/lib/python3.8/site-packages/pip/_internal/cli/base_command.py", line 97, in _inner_run
    return self.run(options, args)
  File "/Applications/anaconda3/lib/python3.8/site-packages/pip/_internal/cli/req_command.py", line 67, in wrapper
    return func(self, options, args)
  File "/Applications/anaconda3/lib/python3.8/site-packages/pip/_internal/commands/install.py", line 484, in run
    installed_versions[distribution.canonical_name] = distribution.version
  File "/Applications/anaconda3/lib/python3.8/site-packages/pip/_internal/metadata/pkg_resources.py", line 192, in version
    return parse_version(self._dist.version)
  File "/Applications/anaconda3/lib/python3.8/site-packages/pip/_vendor/packaging/version.py", line 56, in parse
    return Version(versi

# Imports & global

In [3]:


import pandas as pd
import gradio as gr
from datetime import datetime

# LLM dependencies (we'll initialize the model in a later cell)
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
import torch




# Load your log data

In [4]:
import pandas as pdimport# ----------------------------
uploaded_df = None

# ----------------------------
# Schema normalization (fixes missing log_level/error_type)
# ----------------------------
def normalize_schema(df: pd.DataFrame) -> pd.DataFrame:
    COLUMN_ALIASES = {
        # log level
        "level": "log_level",
        "severity": "log_level",
        "loglevel": "log_level",

        # error type
        "exception": "error_type",
        "error": "error_type",
        "errorType": "error_type",
        "err_type": "error_type",

        # message
        "msg": "message",
        "log_message": "message",

        # service
        "app": "service",
        "application": "service",
        "service_name": "service",
    }

    df = df.copy()

    # Copy aliased columns into canonical columns if needed
    for src, dst in COLUMN_ALIASES.items():
        if src in df.columns and dst not in df.columns:
            df[dst] = df[src]

    # Normalize level values
    if "log_level" in df.columns:
        df["log_level"] = df["log_level"].astype(str).str.upper()

    return df

# ----------------------------
# File loader (CSV / JSONL / XLSX)
# ----------------------------
def load_logs_gradio(file):
    global uploaded_df

    if file is None:
        uploaded_df = None
        return pd.DataFrame({"message": ["No file uploaded yet."]})

    path = file.name

    if path.endswith(".csv"):
        df = pd.read_csv(path)
    elif path.endswith(".jsonl") or path.endswith(".ndjson"):
        df = pd.read_json(path, lines=True)
    elif path.endswith(".xlsx"):
        df = pd.read_excel(path)
    else:
        uploaded_df = None
        return pd.DataFrame({"error": [f"Unsupported file type: {path}"]})

    # Timestamp normalization (support @timestamp or timestamp)
    if "@timestamp" in df.columns:
        df["@timestamp"] = pd.to_datetime(df["@timestamp"], errors="coerce", utc=True)
    elif "timestamp" in df.columns:
        df["timestamp"] = pd.to_datetime(df["timestamp"], errors="coerce", utc=True)

    # ✅ Normalize schema once here
    uploaded_df = normalize_schema(df)

    return uploaded_df.head(15)

# ----------------------------
# Dataset profile (used by UI)
# ----------------------------
def dataset_profile():
    global uploaded_df
    if uploaded_df is None or uploaded_df.empty:
        return "No dataset loaded."
    return (
        f"Loaded ✅ Rows: {len(uploaded_df):,} | Columns: {len(uploaded_df.columns)}\n\n"
        "Columns:\n- " + "\n- ".join(map(str, uploaded_df.columns))
    )


# ----------------------------
# Single source of truth for the dataset


In [5]:
import gradio as gr

with gr.Blocks() as demo:
    gr.Markdown("## Log File Loader (Analyst Demo)")

    with gr.Row():
        with gr.Column(scale=1):
            file_in = gr.File(label="Choose a log file (.csv, .jsonl, .xlsx)")
            profile_btn = gr.Button("Show dataset info")

        with gr.Column(scale=2):
            preview = gr.Dataframe(label="Preview (first rows)")
            info = gr.Markdown()

    file_in.upload(load_logs_gradio, inputs=file_in, outputs=preview)
    profile_btn.click(dataset_profile, outputs=info)

demo.launch()

It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://33e5c1d5ca91dfb776.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


# Normalize & enrich logs (truth layer)

In [6]:
def enrich_logs(df: pd.DataFrame) -> pd.DataFrame:
    """
    Optional enrichment layer.
    Adds derived columns if source fields exist.
    Safe to call multiple times.
    """
    df = df.copy()

    # Timestamp-based enrichment
    if "@timestamp" in df.columns:
        ts = df["@timestamp"]
    elif "timestamp" in df.columns:
        ts = df["timestamp"]
    else:
        ts = None

    if ts is not None:
        df["date"] = ts.dt.date
        df["hour"] = ts.dt.floor("H")

    # HTTP-style status code bucketing
    if "statusCode" in df.columns:
        df["statusClass"] = (df["statusCode"] // 100).astype("Int64").astype(str) + "xx"

    return df


# Analyst‑safe query functions

In [7]:
# ================================AB CELL 5
# Analyst-safe query functions
# ================================

import pandas as pd

def ensure_data_loaded():
    if uploaded_df is None or uploaded_df.empty:
        return False, "❌ No data loaded yet. Please upload a log file first."
    return True, None


def normalize_timestamp(df: pd.DataFrame):
    df = df.copy()

    if "@timestamp" in df.columns:
        df["_ts"] = pd.to_datetime(df["@timestamp"], errors="coerce", utc=True)
    elif "timestamp" in df.columns:
        df["_ts"] = pd.to_datetime(df["timestamp"], errors="coerce", utc=True)
    else:
        df["_ts"] = pd.NaT

    return df


def get_error_summary():
    ok, msg = ensure_data_loaded()
    if not ok:
        return msg

    df = normalize_timestamp(uploaded_df)

    return (
        df[df["log_level"] == "ERROR"]
        .groupby("error_type")
        .size()
        .reset_index(name="error_count")
        .sort_values("error_count", ascending=False)
    )


def get_errors_by_service(service_name):
    ok, msg = ensure_data_loaded()
    if not ok:
        return msg

    df = normalize_timestamp(uploaded_df)

    filtered = df[
        (df["log_level"] == "ERROR") &
        (df["service"] == service_name)
    ]

    if filtered.empty:
        return f"✅ No errors found for service: {service_name}"

    return (
        filtered[["_ts", "service", "error_type", "message", "sourceId"]]
        .rename(columns={"_ts": "timestamp"})
        .sort_values("timestamp", ascending=False)
        .head(50)
    )


def get_errors_in_date_range(start_date, end_date):
    ok, msg = ensure_data_loaded()
    if not ok:
        return msg

    df = normalize_timestamp(uploaded_df)

    start_dt = pd.to_datetime(start_date, errors="coerce", utc=True)
    end_dt = pd.to_datetime(end_date, errors="coerce", utc=True)

    mask = (
        (df["_ts"] >= start_dt) &
        (df["_ts"] <= end_dt) &
        (df["log_level"] == "ERROR")
    )

    result = df.loc[mask]

    if result.empty:
        return "✅ No errors found in the selected date range."

    return (
        result[["_ts", "service", "error_type", "message"]]
        .rename(columns={"_ts": "timestamp"})
        .sort_values("timestamp", ascending=False)
        .head(100)
    )


def get_top_errors(limit=5):
    ok, msg = ensure_data_loaded()
    if not ok:
        return msg

    return (
        uploaded_df[uploaded_df["log_level"] == "ERROR"]
        .groupby(["error_type", "service"])
        .size()
        .reset_index(name="count")
        .sort_values("count", ascending=False)
        .head(int(limit))
    )


def explain_error_type(error_type):
    explanations = {
        "TimeoutException": "The service took too long to respond.",
        "NullPointerException": "The system tried to use missing data.",
        "DatabaseConnectionError": "The service could not reach the database.",
        "AuthFailure": "Authentication or authorization failed.",
        "ServiceUnavailable": "The service was temporarily down.",
    }

    return explanations.get(
        error_type,
        "No predefined explanation available for this error type."
    )


# Load Hugging Face model (small & demo‑friendly)

In [8]:
# ================================ install -U transformers accelerate sentencepiece

import torch
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

MODEL_NAME = "google/flan-t5-small"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME)  # <-- do NOT flip tie_word_embeddings

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)
model.eval()

def llm_invoke(prompt: str, max_new_tokens: int = 256) -> str:
    """Small, reliable 'invoke' for FLAN-T5 without pipelines or LangChain."""
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True).to(device)
    with torch.no_grad():
        out_ids = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,          # deterministic for demo
            num_beams=2,              # a little better quality than greedy, still fast
            no_repeat_ngram_size=3
        )
    return tokenizer.decode(out_ids[0], skip_special_tokens=True)

# Smoke test
print(llm_invoke("Summarize: Service A has TimeoutException spikes after a deployment.", max_new_tokens=80))

# COLAB CELL 6 — FLAN-T5 (version-proof, no HF pipeline registry)
# ================================



config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/308M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/190 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

Service A has a timeoutException spike after a deployment.
